<div style="padding:28px 32px;border-radius:14px;background:linear-gradient(135deg,#1a73e8 0%,#4285f4 45%,#7b61ff 100%);color:#fff">
  <div style="font-size:12px;letter-spacing:2px;text-transform:uppercase;opacity:.85">Gemma 4 Developer Agent · Kaggle</div>
  <div style="font-size:32px;font-weight:700;margin:6px 0 4px">GemmaFineTuning — Training Pipeline</div>
  <div style="font-size:15px;opacity:.92">LoRA fine-tuning of Gemma 4 for a SWE-bench-style, multi-agent coding agent</div>
</div>

<br>

This notebook is the Kaggle half of one local command:

```bash
uv run scripts/kaggle_push.py -training
```

That command pushes this notebook, waits for the run, and downloads `outputs/` plus the full log into `runs/kaggle/` on your machine. All of the training logic lives in the **GemmaFineTuning** repo (`pipeline/`). This notebook only finds the inputs, prepares the GPU environment, calls `PipelineRunner.train()`, and saves a reproducible record of the run.

### Pipeline at a glance

| # | Stage | What happens |
|:-:|:--|:--|
| 0 | **Run configuration** | One `RUN` dict holds every setting; `kaggle_push.py --set key=value` overrides it |
| 1 | **Locate inputs** | Finds the competition data, repo, model and wheelhouse by their contents, not by mount path |
| 2 | **Hardware & stack** | Reports the GPUs, installs the training stack (offline wheelhouse first, then pip), picks bf16 or fp16 |
| 3 | **Load the repo** | Copies the read-only repo dataset to a writable path and imports `PipelineRunner` |
| 4 | **Training data** | Loads the trajectories and prints their size, tool use and outcomes |
| 5 | **Run config** | Fills the repo's YAML with this machine's paths and dtype, and shows every change |
| 6 | **Train** | `PipelineRunner.train()`; everything is logged to `outputs/train.log` |
| 7 | **Artifacts** | Writes the LoRA adapter, `run_manifest.json` and the resolved config to `outputs/` |

### Inputs

| Input | Kaggle source | Why it's needed |
|:--|:--|:--|
| Competition data | `gemma-4-developer-agent` | `tasks.jsonl`, snapshots, graphs and embeddings (the 129 tasks, **held out from training**) |
| Repo | `banwait13/gemmafinetuning` | the public pipeline code (`pipeline/`, configs) |
| Base model | `google/gemma-4` · `transformers` | bf16 QAT weights, the parent of the `*-qat-w4a16-ct` checkpoints we serve |
| Wheelhouse | set in `scripts/kaggle_settings.json` | offline wheels for the training stack (optional when internet is on) |
| Mined data | private dataset (optional) | `DataMining/` output; never part of the public repo |

## 0 · Run configuration

Edit `RUN` here for an interactive session. When you launch from your machine, `kaggle_push.py` writes `--set` values and the local git SHA into `RUN_OVERRIDES`, so the pushed run always records exactly what it was asked to do.

In [ ]:
import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

RUN = {
    "mode": "sft",                                     # "sft" | "dpo"
    "config": "pipeline/configs/train_lora_sft.yaml",  # relative to the repo root
    "model_hint": "e4b-it-qat-q4_0-unquantized",       # picks the attached model whose path contains this
    "train_data": None,                                # None: the config's path; else an absolute path or a file name under /kaggle/input
    "chat_template": None,                             # None: the config's value; else a chat_template.jinja path or a file name under /kaggle/input
    "template_kwargs": None,                           # None: the config's value; e.g. {"enable_thinking": true}, must match serving
    "dry_run": False,                                  # True: check inputs and config, skip training
    "keep_checkpoints": False,                         # False: keep only the final adapter in outputs/
}
RUN_OVERRIDES = {}  # @@RUN_OVERRIDES@@  (rewritten by scripts/kaggle_push.py)
RUN.update(RUN_OVERRIDES)

INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working/outputs")
OUT.mkdir(parents=True, exist_ok=True)
T0 = time.time()

print(json.dumps(RUN, indent=2))

## 1 · Locate the inputs

Kaggle mounts competitions, datasets and models at paths that differ between UI and API attaches and have changed over time. So this step finds each input by a marker it contains:

| Role | Marker |
|:--|:--|
| competition | `tasks.jsonl` + `HARNESS_README.md` |
| repo | `pipeline/main.py` |
| model | `config.json` + `*.safetensors` |
| wheelhouse | `*.whl` files outside the competition folder (the competition's `wheels/` are the task repos' dependencies, not ours) |

In [ ]:
SKIP_DIRS = {"snapshots", "graphs", "embeddings", ".git", "__pycache__"}  # large and never a marker location


def find_dirs(predicate, root=INPUT, max_depth=7):
    """Every directory under root that satisfies predicate(dir, filenames); matches are not descended into."""
    hits = []
    for dirpath, dirnames, filenames in os.walk(root):
        d = Path(dirpath)
        if predicate(d, set(filenames)):
            hits.append(d)
            dirnames[:] = []
            continue
        depth = len(d.relative_to(root).parts)
        dirnames[:] = [] if depth >= max_depth else sorted(n for n in dirnames if n not in SKIP_DIRS)
    return hits


comps = find_dirs(lambda d, f: {"tasks.jsonl", "HARNESS_README.md"} <= f)
repos = find_dirs(lambda d, f: (d / "pipeline" / "main.py").is_file())
models = find_dirs(lambda d, f: "config.json" in f and any(n.endswith(".safetensors") for n in f))
COMP = comps[0] if comps else None
wheelhouses = find_dirs(lambda d, f: any(n.endswith(".whl") for n in f))
WHEELHOUSES = [w for w in wheelhouses if not (COMP and w.is_relative_to(COMP))]

assert repos, "GemmaFineTuning repo not attached: add the banwait13/gemmafinetuning dataset"
REPO_SRC = repos[0]

hint = RUN["model_hint"].lower()
matching = [m for m in models if hint in str(m).lower()] or (models if len(models) == 1 else [])
MODEL = matching[0] if matching else None
if MODEL is None and not RUN["dry_run"]:
    raise FileNotFoundError(f"no attached model matches model_hint={hint!r}; attached models: {[str(m) for m in models]}")

print(f"{'role':<12} path")
print(f"{'-' * 12} {'-' * 60}")
for role, value in [("competition", COMP), ("repo", REPO_SRC), ("model", MODEL), *(("wheelhouse", w) for w in WHEELHOUSES)]:
    print(f"{role:<12} {value or '— not attached —'}")
print()
if len(models) > 1:
    print(f"{len(models)} models attached, using the one matching {hint!r}")
if COMP is None:
    print("⚠ competition data not attached: training works, but the held-out leakage check below is skipped")

## 2 · Hardware & training stack

1. Install every wheel from each attached wheelhouse, offline.
2. `pip install` anything that is still missing (this needs internet on).
3. Pick the precision: **bf16** on Ampere or newer (L4, A100, H100, RTX Pro 6000), **fp16** on T4 and P100, which have no bf16.

In [ ]:
import importlib.metadata
import importlib.util
import subprocess


def sh(cmd):
    """Run a shell command; show its tail and stop the notebook if it fails."""
    print(f"$ {cmd if len(cmd) < 200 else cmd[:200] + ' …'}")
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode:
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
        raise RuntimeError(f"command failed ({r.returncode})")
    return r.stdout


GPU_INFO = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader").strip().splitlines()
for i, g in enumerate(GPU_INFO):
    print(f"  GPU{i}: {g}")

for wh in WHEELHOUSES:
    wheels = " ".join(f'"{p}"' for p in sorted(wh.glob("*.whl")))
    sh(f'pip install -q --no-deps --no-index --find-links "{wh}" {wheels}')

NEEDED = {"transformers": "transformers", "peft": "peft", "trl": "trl", "accelerate": "accelerate",
          "datasets": "datasets", "bitsandbytes": "bitsandbytes", "yaml": "pyyaml"}
missing = [pkg for mod, pkg in NEEDED.items() if importlib.util.find_spec(mod) is None]
if missing:
    sh("pip install -q " + " ".join(missing))

import torch

BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
VERSIONS = {pkg: importlib.metadata.version(pkg) for pkg in ["torch", *NEEDED.values()]}

print()
print(f"precision  {'bf16' if BF16 else 'fp16'}   ({torch.cuda.device_count()} GPU, CUDA {torch.version.cuda})")
for pkg, v in VERSIONS.items():
    print(f"  {pkg:<14} {v}")

## 3 · Load the GemmaFineTuning repo

`/kaggle/input` is read-only, and the trainer resolves relative paths from the working directory. So the repo is copied to `/tmp/GemmaFineTuning` and the notebook runs from there. Run outputs go to `/kaggle/working/outputs`, the only folder Kaggle saves.

In [ ]:
import shutil

REPO = Path("/tmp/GemmaFineTuning")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(REPO_SRC, REPO, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".git"))
os.chdir(REPO)
sys.path.insert(0, str(REPO))

from pipeline.main import PipelineRunner

runner = PipelineRunner(REPO)
print(f"repo  {REPO_SRC}  →  {REPO}")
print(f"git   {RUN.get('git_sha', 'unknown (interactive run)')}")
print()
runner.status();

## 4 · Training data

Mined trajectories come from a **private** dataset, which you point to with `--set train_data=<file name>`. With no override, the config's path inside the repo is used, and in the public repo that file is only a placeholder sample.

**Leakage guard:** the 129 competition tasks are our held-out test set. If any training trajectory carries one of their task IDs, the run stops here.

In [ ]:
from collections import Counter

import pandas as pd
import yaml

BASE_CFG = yaml.safe_load((REPO / RUN["config"]).read_text(encoding="utf-8"))


def resolve_input(value, default=None):
    """None: the default; an absolute path as is; else the first file with that name under /kaggle/input."""
    if value is None:
        return default
    if Path(value).is_absolute():
        return Path(value)
    return next((p for p in INPUT.rglob(value) if "snapshots" not in p.parts), None)


TRAIN_DATA = resolve_input(RUN["train_data"], REPO / BASE_CFG["dataset"]["train_data_path"])
assert TRAIN_DATA and TRAIN_DATA.is_file(), f"training data not found: {RUN['train_data'] or TRAIN_DATA}"

rows = [json.loads(line) for line in TRAIN_DATA.read_text(encoding="utf-8").splitlines() if line.strip()]

held_out = set()
if COMP:
    held_out = {json.loads(line)["instance_id"] for line in (COMP / "tasks.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()}
leaked = sorted({r.get("instance_id") or r.get("task_id") for r in rows} & held_out)
assert not leaked, f"{len(leaked)} held-out competition tasks found in training data: {leaked[:5]}"

tools = Counter(tc["function"]["name"] for r in rows for m in r.get("messages", []) for tc in m.get("tool_calls") or [])
summary = pd.DataFrame([{
    "trajectories": len(rows),
    "resolved": sum(bool(r.get("resolved")) for r in rows),
    "with tool schemas": sum(bool(r.get("tools")) for r in rows),
    "repos": len({r.get("repo") for r in rows}),
    "median turns": pd.Series([len(r.get("messages", [])) for r in rows]).median(),
    "median chars": pd.Series([sum(len(str(m.get("content") or "")) + len(m.get("reasoning") or "") for m in r.get("messages", [])) for r in rows]).median(),
}], index=[TRAIN_DATA.name])
display(summary)
display(pd.Series(tools, name="tool calls").sort_values(ascending=False).to_frame())

print(f"leakage check: {'passed' if COMP else 'skipped'} ({len(held_out)} held-out task IDs)")
if TRAIN_DATA.is_relative_to(REPO) and len(rows) < 10:
    print("⚠ this is the repo's placeholder sample: attach the private mined-data dataset and pass --set train_data=<file>")

## 5 · Resolve the run config

The repo's YAML is the single source of hyperparameters. This step only swaps in what depends on this machine (the model path, data path, output directory and precision), then writes the result to `outputs/run_config.yaml`. That file is the exact config the trainer reads.

In [ ]:
import copy

dtype = "bfloat16" if BF16 else "float16"
CKPT_DIR = OUT / "checkpoints"
CFG = copy.deepcopy(BASE_CFG)
CFG["model"].update(base_model_name_or_path=str(MODEL), torch_dtype=dtype, bnb_4bit_compute_dtype=dtype)
CFG["dataset"]["train_data_path"] = str(TRAIN_DATA)
CFG["training"].update(output_dir=str(CKPT_DIR), bf16=BF16, fp16=False)

# Loss is masked from the rendered chat template, so it must be the template the agent is served with.
if RUN["chat_template"] is not None:
    CHAT_TEMPLATE = resolve_input(RUN["chat_template"])
    assert CHAT_TEMPLATE and CHAT_TEMPLATE.is_file(), f"chat template not found: {RUN['chat_template']}"
    CFG["dataset"]["chat_template_path"] = str(CHAT_TEMPLATE)
if RUN["template_kwargs"] is not None:
    CFG["dataset"]["chat_template_kwargs"] = RUN["template_kwargs"]

RUN_CFG = OUT / "run_config.yaml"
RUN_CFG.write_text(yaml.safe_dump(CFG, sort_keys=False), encoding="utf-8")

changes = [
    {"key": f"{section}.{key}", "repo default": BASE_CFG[section].get(key), "this run": value}
    for section in CFG for key, value in CFG[section].items()
    if BASE_CFG.get(section, {}).get(key) != value
]
display(pd.DataFrame(changes).set_index("key"))
template = CFG["dataset"].get("chat_template_path") or "the training checkpoint's own"
print(f"chat template: {template}  ·  kwargs {CFG['dataset'].get('chat_template_kwargs') or {}}")
print(f"written to {RUN_CFG}")

## 6 · Train

Everything the pipeline logs goes both to this output and to `outputs/train.log`. If training fails, the error is recorded rather than stopping the notebook here. Step 7 then still writes the manifest, and fails the run at the very end so Kaggle reports it as an error.

In [ ]:
import logging
import traceback

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-7s %(name)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(OUT / "train.log", encoding="utf-8")],
    force=True,
)

STATUS, TRAIN_ERROR, ADAPTER = "skipped (dry_run)", None, None
t_train = time.time()
if not RUN["dry_run"]:
    try:
        # Data, model and output dir are passed explicitly: they override the YAML, which stays a record of the run.
        ADAPTER = runner.train(mode=RUN["mode"], config_path=RUN_CFG,
                               train_data=TRAIN_DATA, base_model=MODEL, output_dir=CKPT_DIR)
        STATUS = "completed"
    except Exception:
        STATUS, TRAIN_ERROR = "failed", traceback.format_exc()
        logging.getLogger("notebook").error("training failed: %s", TRAIN_ERROR)
TRAIN_SECONDS = round(time.time() - t_train)
print()
print(f"{STATUS} in {TRAIN_SECONDS // 60} min {TRAIN_SECONDS % 60} s")

## 7 · Save artifacts

`outputs/` is what `kaggle_push.py` downloads into `runs/kaggle/<run>/`:

| File | Contents |
|:--|:--|
| `checkpoints/final_adapter/` | the LoRA adapter (`adapter_model.safetensors` + `adapter_config.json`), ready for `--action package` |
| `run_manifest.json` | git SHA, `RUN`, inputs, model, GPUs, package versions, data stats, timings, status |
| `run_config.yaml` | the exact config the trainer used |
| `train.log` | the pipeline's log |

In [ ]:
ADAPTER = Path(ADAPTER) if ADAPTER else CKPT_DIR / "final_adapter"
if not RUN["keep_checkpoints"]:
    for ckpt in CKPT_DIR.glob("checkpoint-*"):
        shutil.rmtree(ckpt)

MANIFEST = {
    "status": STATUS,
    "error": TRAIN_ERROR,
    "git_sha": RUN.get("git_sha"),
    "run": RUN,
    "inputs": {"competition": str(COMP) if COMP else None, "repo": str(REPO_SRC),
               "model": str(MODEL) if MODEL else None, "wheelhouses": [str(w) for w in WHEELHOUSES]},
    "train_data": {"path": str(TRAIN_DATA), "trajectories": len(rows), "held_out_checked": len(held_out)},
    "chat_template": {"path": CFG["dataset"].get("chat_template_path"), "kwargs": CFG["dataset"].get("chat_template_kwargs")},
    "gpus": GPU_INFO,
    "precision": "bf16" if BF16 else "fp16",
    "packages": VERSIONS,
    "adapter": str(ADAPTER) if ADAPTER.exists() else None,
    "train_seconds": TRAIN_SECONDS,
    "total_seconds": round(time.time() - T0),
    "finished_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
(OUT / "run_manifest.json").write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")

files = [{"file": str(p.relative_to(OUT)), "MB": round(p.stat().st_size / 1e6, 2)} for p in sorted(OUT.rglob("*")) if p.is_file()]
display(pd.DataFrame(files).set_index("file"))
print(f"status {STATUS}  ·  total {MANIFEST['total_seconds'] // 60} min")

if TRAIN_ERROR:
    raise RuntimeError("training failed; the traceback is in outputs/train.log and outputs/run_manifest.json")

---

### From your machine

```bash
uv run scripts/kaggle_push.py -training                       # push, wait, download outputs + log
uv run scripts/kaggle_push.py -training --set dry_run=true    # smoke test: inputs + config only
uv run scripts/kaggle_push.py -training --no-wait             # push and return
uv run scripts/kaggle_push.py -training -status               # check the run
uv run scripts/kaggle_push.py -training -output               # download outputs + log now
```

Before pushing, commit, push to GitHub and update the **GemmaFineTuning** dataset. The script stamps the run with your local git SHA and warns if the working tree differs from what you pushed.

<sub>Runs land in <code>runs/kaggle/&lt;slug&gt;_&lt;timestamp&gt;/</code> (gitignored): <code>outputs/</code>, the raw Kaggle log, and a plain-text <code>run.log</code>.</sub>